In [3]:
!pip install youtube-transcript-api


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi

print("Library Installed Successfully")

Library Installed Successfully


In [6]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "Rni7Fz7208c"

ytt_api = YouTubeTranscriptApi()

transcript = ytt_api.fetch(video_id)

print("Total transcript entries:", len(transcript))

print("\nFirst 5 transcript entries:\n")

for item in transcript[:5]:
    print(item)

Total transcript entries: 2839

First 5 transcript entries:

FetchedTranscriptSnippet(text='Heat. Heat.', start=0.0, duration=3.0)
FetchedTranscriptSnippet(text='[music]', start=11.79, duration=2.02)
FetchedTranscriptSnippet(text='[music]', start=21.615, duration=4.504)
FetchedTranscriptSnippet(text='Our', start=23.119, duration=3.0)
FetchedTranscriptSnippet(text='[music]', start=27.51, duration=2.02)


In [8]:
from youtube_transcript_api import YouTubeTranscriptApi

print(dir(YouTubeTranscriptApi))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'fetch', 'list']


In [9]:
from youtube_transcript_api import YouTubeTranscriptApi

api = YouTubeTranscriptApi()

transcript = api.fetch("Rni7Fz7208c")

print(type(transcript))
print("Transcript fetched successfully!")

print("\nFirst entry:\n")
print(transcript[0])

<class 'youtube_transcript_api._transcripts.FetchedTranscript'>
Transcript fetched successfully!

First entry:

FetchedTranscriptSnippet(text='Heat. Heat.', start=0.0, duration=3.0)


In [10]:
from youtube_transcript_api import YouTubeTranscriptApi

api = YouTubeTranscriptApi()

transcript = api.fetch("Rni7Fz7208c")

with open("transcript.txt", "w", encoding="utf-8") as f:
    for item in transcript:
        f.write(f"[{item.start:.2f}] {item.text}\n")

print("Transcript saved to transcript.txt")
print("Total transcript entries:", len(transcript))

Transcript saved to transcript.txt
Total transcript entries: 2839


In [11]:
chunks = []
chunk_size = 20

for i in range(0, len(transcript), chunk_size):
    chunk = transcript[i:i + chunk_size]

    start_time = chunk[0].start

    text = " ".join([entry.text for entry in chunk])

    chunks.append({
        "timestamp": start_time,
        "text": text
    })

print("Total Chunks Created:", len(chunks))

print("\nFirst Chunk:\n")
print(chunks[0])

Total Chunks Created: 142

First Chunk:

{'timestamp': 0.0, 'text': "Heat. Heat. [music] [music] Our [music] [music] [music] audience is largely wannabe entrepreneurs in India. And I feel like all of us have so much to learn from you because you've done it so many times over in so many different domains. [music] >> Yeah. >> Uh so we will speak to them today and I will try and center all my questions in that direction so they can take advantage of this conversation and maybe start [music] take a chance and build something."}


In [12]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.8 MB/s eta 0:00:00


In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [14]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Total Embeddings:", len(embeddings))
print("Embedding Dimension:", len(embeddings[0]))

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Total Embeddings: 142
Embedding Dimension: 384


In [15]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(142, 384)


In [16]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS Index Created Successfully!")
print("Total Vectors Stored:", index.ntotal)

FAISS Index Created Successfully!
Total Vectors Stored: 142


In [17]:
query = "What is first principles thinking?"

query_embedding = model.encode([query]).astype("float32")

distances, indices = index.search(query_embedding, k=3)

print("Top Matching Chunks:\n")

for idx in indices[0]:
    print("=" * 80)
    print("Timestamp:", chunks[idx]["timestamp"])
    print(chunks[idx]["text"][:500])

Top Matching Chunks:

Timestamp: 3622.96
uh, you know, like what what do you want to major in type of thing. Uh [laughter] it's usually like they you get you get given a religion by your parents and your community. Um, so you know, um, but you know, I mean, I think, you know, there's there's good things in in in in in all religions that are good principles um that you you can sort of read any religious text and say, "Okay, this is a good principle. This is going to be this is going to lead to a better society most likely, you know." Um
Timestamp: 6704.56
So, that's that's the the main thing you should aim for. Aim to make more than you take. Um be a be a you know a net contributor to to society. Um it's and and it's it's kind of like the pursuit of happiness. You know, you if you want to create something valuable financially, you you don't pursue that. You you it's best to actually pursue make providing useful products and services. If you do that, then money will come as a natural con

In [18]:
def ask_podcast(question, top_k=3):

    query_embedding = model.encode([question]).astype("float32")

    distances, indices = index.search(query_embedding, k=top_k)

    best_idx = indices[0][0]

    result = chunks[best_idx]

    print("="*80)
    print("QUESTION:")
    print(question)

    print("\nBEST TIMESTAMP:")
    print(result["timestamp"])

    print("\nRELEVANT TRANSCRIPT:")
    print(result["text"][:1500])

    return result

In [19]:
ask_podcast("What advice does Elon Musk give to entrepreneurs?")

QUESTION:
What advice does Elon Musk give to entrepreneurs?

BEST TIMESTAMP:
1198.96

RELEVANT TRANSCRIPT:
daily fluctuations, >> right? What's got you most excited now, Elon, in terms of all that you're building? You're doing so much. So, let me just preface and contextualize who is watching this. Uh, our audience is largely wannabe entrepreneurs in India. >> Okay. uh really ambitious, really hungry, want to take the risk and build something and I feel like all of us have so much to learn from you because you've done it so many times over in so many different domains. >> Yeah. >> Uh so we will speak to them today and I will try and center all my questions in


{'timestamp': 1198.96,
 'text': "daily fluctuations, >> right? What's got you most excited now, Elon, in terms of all that you're building? You're doing so much. So, let me just preface and contextualize who is watching this. Uh, our audience is largely wannabe entrepreneurs in India. >> Okay. uh really ambitious, really hungry, want to take the risk and build something and I feel like all of us have so much to learn from you because you've done it so many times over in so many different domains. >> Yeah. >> Uh so we will speak to them today and I will try and center all my questions in"}

In [20]:
def seconds_to_timestamp(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)

    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

In [21]:
print(seconds_to_timestamp(3622.96))

01:00:22


In [22]:
ask_podcast("What advice does Elon Musk give to entrepreneurs?")

QUESTION:
What advice does Elon Musk give to entrepreneurs?

BEST TIMESTAMP:
1198.96

RELEVANT TRANSCRIPT:
daily fluctuations, >> right? What's got you most excited now, Elon, in terms of all that you're building? You're doing so much. So, let me just preface and contextualize who is watching this. Uh, our audience is largely wannabe entrepreneurs in India. >> Okay. uh really ambitious, really hungry, want to take the risk and build something and I feel like all of us have so much to learn from you because you've done it so many times over in so many different domains. >> Yeah. >> Uh so we will speak to them today and I will try and center all my questions in


{'timestamp': 1198.96,
 'text': "daily fluctuations, >> right? What's got you most excited now, Elon, in terms of all that you're building? You're doing so much. So, let me just preface and contextualize who is watching this. Uh, our audience is largely wannabe entrepreneurs in India. >> Okay. uh really ambitious, really hungry, want to take the risk and build something and I feel like all of us have so much to learn from you because you've done it so many times over in so many different domains. >> Yeah. >> Uh so we will speak to them today and I will try and center all my questions in"}

In [23]:
!pip install transformers torch accelerate sentencepiece

In [26]:
pipeline("text-generation")

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

TextGenerationPipeline: {'model': 'GPT2LMHeadModel', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text', 'output_modalities': ('text',)}

In [27]:
def ask_podcast(question, top_k=3):

    query_embedding = model.encode([question]).astype("float32")

    distances, indices = index.search(query_embedding, k=top_k)

    best_idx = indices[0][0]

    result = chunks[best_idx]

    timestamp = result["timestamp"]

    hours = int(timestamp // 3600)
    minutes = int((timestamp % 3600) // 60)
    seconds = int(timestamp % 60)

    readable_time = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    youtube_link = f"https://www.youtube.com/watch?v=Rni7Fz7208c&t={int(timestamp)}s"

    print("="*80)
    print("QUESTION:")
    print(question)

    print("\nTIMESTAMP:")
    print(readable_time)

    print("\nVIDEO LINK:")
    print(youtube_link)

    print("\nANSWER:")
    print(result["text"][:1000])

    return result

In [28]:
ask_podcast("What advice does Elon Musk give to entrepreneurs?")

QUESTION:
What advice does Elon Musk give to entrepreneurs?

TIMESTAMP:
00:19:58

VIDEO LINK:
https://www.youtube.com/watch?v=Rni7Fz7208c&t=1198s

ANSWER:
daily fluctuations, >> right? What's got you most excited now, Elon, in terms of all that you're building? You're doing so much. So, let me just preface and contextualize who is watching this. Uh, our audience is largely wannabe entrepreneurs in India. >> Okay. uh really ambitious, really hungry, want to take the risk and build something and I feel like all of us have so much to learn from you because you've done it so many times over in so many different domains. >> Yeah. >> Uh so we will speak to them today and I will try and center all my questions in


{'timestamp': 1198.96,
 'text': "daily fluctuations, >> right? What's got you most excited now, Elon, in terms of all that you're building? You're doing so much. So, let me just preface and contextualize who is watching this. Uh, our audience is largely wannabe entrepreneurs in India. >> Okay. uh really ambitious, really hungry, want to take the risk and build something and I feel like all of us have so much to learn from you because you've done it so many times over in so many different domains. >> Yeah. >> Uh so we will speak to them today and I will try and center all my questions in"}

In [29]:
def ask_podcast(question, top_k=5):

    query_embedding = model.encode([question]).astype("float32")

    distances, indices = index.search(query_embedding, k=top_k)

    print("="*80)
    print("QUESTION:", question)

    print("\nTOP MATCHES\n")

    for rank, idx in enumerate(indices[0], start=1):

        timestamp = chunks[idx]["timestamp"]

        hours = int(timestamp // 3600)
        minutes = int((timestamp % 3600) // 60)
        seconds = int(timestamp % 60)

        readable_time = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

        print("\n" + "="*80)
        print(f"Match #{rank}")
        print("Timestamp:", readable_time)
        print("Video Link:",
              f"https://www.youtube.com/watch?v=Rni7Fz7208c&t={int(timestamp)}s")

        print("\nTranscript:")
        print(chunks[idx]["text"][:500])

In [30]:
ask_podcast("What advice does Elon Musk give to entrepreneurs?")

QUESTION: What advice does Elon Musk give to entrepreneurs?

TOP MATCHES


Match #1
Timestamp: 00:19:58
Video Link: https://www.youtube.com/watch?v=Rni7Fz7208c&t=1198s

Transcript:
daily fluctuations, >> right? What's got you most excited now, Elon, in terms of all that you're building? You're doing so much. So, let me just preface and contextualize who is watching this. Uh, our audience is largely wannabe entrepreneurs in India. >> Okay. uh really ambitious, really hungry, want to take the risk and build something and I feel like all of us have so much to learn from you because you've done it so many times over in so many different domains. >> Yeah. >> Uh so we will speak

Match #2
Timestamp: 00:18:33
Video Link: https://www.youtube.com/watch?v=Rni7Fz7208c&t=1113s

Transcript:
um so >> my primary job elon is a stock broker and stock investor Okay. >> There is no predictive value. Nobody knows what will happen tomorrow. >> Well, but I think you can generally say, you know, um that um i

In [31]:
!pip install gradio

In [32]:
import gradio as gr

def podcast_bot(question):

    query_embedding = model.encode([question]).astype("float32")

    distances, indices = index.search(query_embedding, k=5)

    best_idx = indices[0][0]

    result = chunks[best_idx]

    timestamp = result["timestamp"]

    hours = int(timestamp // 3600)
    minutes = int((timestamp % 3600) // 60)
    seconds = int(timestamp % 60)

    readable_time = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    youtube_link = f"https://www.youtube.com/watch?v=Rni7Fz7208c&t={int(timestamp)}s"

    return f"""
Answer:

{result['text'][:800]}

Timestamp:
{readable_time}

Video Link:
{youtube_link}
"""

demo = gr.Interface(
    fn=podcast_bot,
    inputs="text",
    outputs="text",
    title="Elon Musk Podcast Q&A Bot",
    description="Ask questions about the podcast and receive answers with timestamps."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://652eb9275da5609fd6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
best_idx = indices[0][0]

In [34]:
def podcast_bot(question):

    query_embedding = model.encode([question]).astype("float32")

    distances, indices = index.search(query_embedding, k=5)

    best_idx = indices[0][0]

    result = chunks[best_idx]

    timestamp = result["timestamp"]

    hours = int(timestamp // 3600)
    minutes = int((timestamp % 3600) // 60)
    seconds = int(timestamp % 60)

    readable_time = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    youtube_link = f"https://www.youtube.com/watch?v=Rni7Fz7208c&t={int(timestamp)}s"

    answer = result["text"][:700]

    return f"""
Answer:
{answer}

Timestamp:
{readable_time}

Video Link:
{youtube_link}
"""

In [35]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://652eb9275da5609fd6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [36]:
best_idx = indices[0][0]

In [37]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://652eb9275da5609fd6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [38]:
import gradio as gr

demo = gr.Interface(
    fn=podcast_bot,
    inputs=gr.Textbox(
        label="Ask a Question",
        lines=2
    ),
    outputs=gr.Textbox(
        label="Answer",
        lines=15
    ),
    title="Elon Musk Podcast Q&A Bot",
    description="Ask questions about the podcast and receive answers with timestamps."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e98e455906443e69e7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [39]:
def podcast_bot(question):

    query_embedding = model.encode([question]).astype("float32")
    distances, indices = index.search(query_embedding, k=5)

    best_idx = indices[0][0]
    result = chunks[best_idx]

    timestamp = result["timestamp"]

    hours = int(timestamp // 3600)
    minutes = int((timestamp % 3600) // 60)
    seconds = int(timestamp % 60)

    readable_time = f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    youtube_link = f"https://www.youtube.com/watch?v=Rni7Fz7208c&t={int(timestamp)}s"

    answer = result["text"][:1000]

    return answer, readable_time, youtube_link


demo = gr.Interface(
    fn=podcast_bot,
    inputs=gr.Textbox(label="Question"),
    outputs=[
        gr.Textbox(label="Answer", lines=12),
        gr.Textbox(label="Timestamp"),
        gr.Textbox(label="YouTube Link")
    ],
    title="Elon Musk Podcast Q&A Bot"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2e8c4549d4cb14545a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [40]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2e8c4549d4cb14545a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
